# BoolQ Comparative Study

BoolQ is a Q/A dataset, containing 12697 examples sourced from google queries.

Each example is triplet of (question, passage, answer)

**Authors:** Anh Nguyen, Berke Kara

#Pre-Process

In [1]:
from datasets import load_dataset,concatenate_datasets,Dataset, DatasetDict
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


ds = load_dataset("fancyzhx/ag_news")
ds_combined = concatenate_datasets([ds["train"], ds["test"]])
df = pd.DataFrame(ds_combined)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [2]:
print(torch.cuda.is_available())

True


In [3]:
df_sampled, _ = train_test_split(
    df,
    train_size=30000,
    random_state=42,
    stratify=df["label"]
)

train_df, temp_df = train_test_split(
    df_sampled,
    test_size=0.3,
    random_state=42,
    stratify=df_sampled["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

In [4]:
X_train = train_df['text']
y_train = train_df['label']

X_val = val_df['text']
y_val = val_df['label']

X_test = test_df['text']
y_test = test_df['label']

print(f"Training examples: {len(X_train)}")
print(f"Validation examples: {len(X_val)}")
print(f"Test examples: {len(X_test)}")

Training examples: 21000
Validation examples: 4500
Test examples: 4500


# Model 1: Classical Baseline

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=20000)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_vec, y_train)

val_preds = model.predict(X_val_vec)
print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print("Validation Macro F1:", f1_score(y_val, val_preds, average="macro"))

test_preds = model.predict(X_test_vec)
print("Test Accuracy:", accuracy_score(y_test, test_preds))
print("Test Macro F1:", f1_score(y_test, test_preds, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, test_preds))

Validation Accuracy: 0.8931111111111111
Validation Macro F1: 0.8927267289679393
Test Accuracy: 0.9035555555555556
Test Macro F1: 0.9031899730813575

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.88      0.90      1125
           1       0.94      0.98      0.96      1125
           2       0.89      0.87      0.88      1125
           3       0.87      0.89      0.88      1125

    accuracy                           0.90      4500
   macro avg       0.90      0.90      0.90      4500
weighted avg       0.90      0.90      0.90      4500



# Model 4: Pretrained Model with Fine-Tuning

In [6]:
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          Trainer, TrainingArguments, DataCollatorWithPadding)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False),
    "validation": Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False),
    "test": Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False),
})

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./distilbert_agnews",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

trainer.train()

predictions = trainer.predict(tokenized_dataset["validation"])

logits = predictions.predictions
labels = predictions.label_ids

# val results
preds = np.argmax(logits, axis=1)

accuracy = accuracy_score(labels, preds)
macro_f1 = f1_score(labels, preds, average="macro")

print("Validation Accuracy:", accuracy)
print("Validation Macro F1:", macro_f1)

# test results

test_preds = trainer.predict(tokenized_dataset["test"])

preds = np.argmax(test_preds.predictions, axis=1)
labels = test_preds.label_ids

print("Test Accuracy:", accuracy_score(labels, preds))
print("Test Macro F1:", f1_score(labels, preds, average="macro"))









config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Validation Accuracy: 0.9235555555555556
Validation Macro F1: 0.9234319849636013


Test Accuracy: 0.9286666666666666
Test Macro F1: 0.928676643894495


# Model 5: pretrained model without task-specific training